# NimbusPay ticket triage

Run on a free Colab T4. Upload `candidate_pack_colab.zip` to MyDrive. Set `RUN_EXPERIMENTS=True` for comparisons; leave it `False` for the final reproduction run. Restart the session after changing installed packages in an already used runtime.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.51.3', 'peft==0.15.2',
                'accelerate==1.6.0', 'bitsandbytes==0.50.2'], check=True)


In [ ]:
import gc, json, os, shutil, sys, zipfile
from pathlib import Path
import torch
import transformers, bitsandbytes
from google.colab import drive

print('Transformers:', transformers.__version__, 'bitsandbytes:', bitsandbytes.__version__)
assert transformers.__version__ == '4.51.3'
assert bitsandbytes.__version__ == '0.50.2'
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert 'T4' in torch.cuda.get_device_name(0)
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__)
bnb_cuda = torch.version.cuda.replace('.', '')
bnb_binary = Path(bitsandbytes.__file__).parent / f'libbitsandbytes_cuda{bnb_cuda}.so'
assert bnb_binary.exists(), f'Missing bitsandbytes CUDA binary: {bnb_binary.name}'
print('bitsandbytes binary:', bnb_binary.name)

drive.mount('/content/drive')
ARCHIVE = Path('/content/drive/MyDrive/candidate_pack_colab.zip')
WORK = Path('/content/candidate_pack')
assert ARCHIVE.exists(), f'Upload {ARCHIVE.name} to MyDrive first'
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    archive.extractall(WORK)
os.chdir(WORK)


## Audit and clean training data


In [ ]:
import csv
import json
import re
from collections import Counter
from datetime import datetime
from pathlib import Path


ROOT = Path("/content/candidate_pack")
DATA = ROOT / "data"
KEYS = (
    "category", "priority", "amount_paise", "txn_date", "txn_id",
    "channel", "language", "needs_human",
)
CATEGORIES = {"payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"}
CHANNELS = {"upi", "card", "netbanking", "wallet"}
LEGAL = re.compile(r"\b(?:RBI|ombudsman|consumer court|lawyer|legal notice|legal action)\b", re.I)
TXN_ID = re.compile(r"NP\d{10}")
OLD_DATE = re.compile(r"\d{2}/\d{2}/\d{4}")


def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def ticket(row):
    return row["messages"][1]["content"]


def active_ticket_text(row):
    return "\n".join(line for line in ticket(row).splitlines() if not line.lstrip().startswith(">"))


def expected_priority(row, label):
    category = label["category"]
    amount = label["amount_paise"]
    if category == "fraud" or (type(amount) is int and amount >= 5_000_000) or LEGAL.search(active_ticket_text(row)):
        return "P1"
    if category == "account_access" or (category in {"payment_failed", "refund"} and type(amount) is int and amount >= 500_000):
        return "P2"
    return "P3"


def valid_label(label):
    if set(label) != set(KEYS):
        return False
    if label["category"] not in CATEGORIES or label["priority"] not in {"P1", "P2", "P3"}:
        return False
    if type(label["amount_paise"]) not in (int, type(None)):
        return False
    if label["txn_date"] is not None:
        try:
            if datetime.strptime(label["txn_date"], "%Y-%m-%d").strftime("%Y-%m-%d") != label["txn_date"]:
                return False
        except (TypeError, ValueError):
            return False
    if label["txn_id"] is not None and (type(label["txn_id"]) is not str or not TXN_ID.fullmatch(label["txn_id"])):
        return False
    if label["channel"] is not None and label["channel"] not in CHANNELS:
        return False
    if label["language"] not in {"en", "hi-en"} or type(label["needs_human"]) is not bool:
        return False
    if label["category"] in {"account_access", "kyc", "other"}:
        if any(label[key] is not None for key in ("amount_paise", "txn_date", "txn_id", "channel")):
            return False
    if label["priority"] == "P1" and not label["needs_human"]:
        return False
    return True


def prepare_data():
    train = read_jsonl(DATA / "train.jsonl")
    dev = read_jsonl(DATA / "dev.jsonl")
    dev_texts = {ticket(row).strip() for row in dev}
    raw_safe = []
    cleaned = []
    changes = []
    issues = Counter()
    seen_clean_texts = set()

    for row in train:
        reasons = []
        text = ticket(row).strip()
        if text in dev_texts:
            issues["train/dev overlap"] += 1
            changes.append((row["id"], "drop", "ticket text overlaps dev; excluded from all training runs"))
            continue
        raw_safe.append(row)

        try:
            label = json.loads(row["messages"][-1]["content"])
        except json.JSONDecodeError:
            issues["malformed label JSON"] += 1
            changes.append((row["id"], "drop", "assistant label is incomplete or invalid JSON"))
            continue
        if not isinstance(label, dict):
            issues["non-object label"] += 1
            changes.append((row["id"], "drop", "assistant label is not a JSON object"))
            continue

        label = label.copy()
        if "prio" in label and "priority" not in label:
            label.pop("prio")
            issues["legacy prio field"] += 1
            reasons.append("replace legacy prio field with schema priority")

        if label.get("txn_date") is not None and OLD_DATE.fullmatch(str(label["txn_date"])):
            label["txn_date"] = datetime.strptime(label["txn_date"], "%d/%m/%Y").strftime("%Y-%m-%d")
            issues["legacy transaction date"] += 1
            reasons.append("convert legacy transaction date to ISO format")

        old_id = label.get("txn_id")
        if isinstance(old_id, str) and not TXN_ID.fullmatch(old_id):
            normalized = re.sub(r"[^A-Za-z0-9]", "", old_id).upper()
            if TXN_ID.fullmatch(normalized):
                label["txn_id"] = normalized
                issues["unnormalized transaction ID"] += 1
                reasons.append("normalize NimbusPay transaction ID")

        if "category" in label and "amount_paise" in label:
            correct_priority = expected_priority(row, label)
            if label.get("priority") != correct_priority:
                label["priority"] = correct_priority
                issues["wrong or missing priority"] += 1
                reasons.append("recompute priority from category, labelled amount, and active legal-threat text")
            if correct_priority == "P1" and label.get("needs_human") is False:
                label["needs_human"] = True
                issues["P1 needs_human mismatch"] += 1
                reasons.append("set needs_human true for P1")

        if not valid_label(label):
            issues["unresolved schema violation"] += 1
            changes.append((row["id"], "drop", "remaining label violates current schema after safe fixes"))
            continue
        if text in seen_clean_texts:
            issues["duplicate clean ticket"] += 1
            changes.append((row["id"], "drop", "duplicate ticket text and label in training set"))
            continue
        seen_clean_texts.add(text)

        fixed = dict(row)
        fixed["messages"] = [dict(message) for message in row["messages"]]
        fixed["messages"][-1]["content"] = json.dumps(label, ensure_ascii=False)
        cleaned.append(fixed)
        if reasons:
            changes.append((row["id"], "fix", "; ".join(reasons)))

    write_jsonl(DATA / "train_raw_safe.jsonl", raw_safe)
    write_jsonl(DATA / "train_clean.jsonl", cleaned)
    with (ROOT / "cleaning_log.csv").open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(("id", "action", "reason"))
        writer.writerows(changes)

    print(f"Original train: {len(train)}; raw leakage-safe: {len(raw_safe)}; clean: {len(cleaned)}")
    print(f"Logged changes: {len(changes)}")
    for issue, count in sorted(issues.items()):
        print(f"  {issue}: {count}")
    assert not ({ticket(row).strip() for row in cleaned} & dev_texts)
    assert all(valid_label(json.loads(row["messages"][-1]["content"])) for row in cleaned)


prepare_data()


## Scorer and training functions


In [ ]:
from collections import defaultdict

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def same(a, b):
    return type(a) is type(b) and a == b


def parse(text):
    try:
        obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError):
        return None
    return obj if isinstance(obj, dict) else None


def score(gold_rows, pred_rows):
    preds = {p["id"]: p.get("output") for p in pred_rows}
    groups = defaultdict(lambda: {"n": 0, "valid": 0, "exact": 0, "field": defaultdict(int)})
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = parse(preds.get(g["id"]))
        names = ["all"] + ([f"slice={g['slice']}"] if "slice" in g else []) + [f"category={gold['category']}"]
        hits = {k: pred is not None and k in pred and same(pred[k], gold[k]) for k in KEYS}
        exact = pred is not None and all(hits.values()) and set(pred) == set(KEYS)
        for name in names:
            s = groups[name]
            s["n"] += 1
            s["valid"] += pred is not None
            s["exact"] += exact
            for k in KEYS:
                s["field"][k] += hits[k]
    out = {}
    for name, s in groups.items():
        n = s["n"]
        fields = {k: s["field"][k] / n for k in KEYS}
        out[name] = {"n": n, "json_valid": s["valid"] / n, "exact_match": s["exact"] / n,
                     "mean_field_acc": sum(fields.values()) / len(KEYS), "fields": fields}
    return out



In [ ]:
import json
import math
import random
import time
from pathlib import Path

import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup



ROOT = Path("/content/candidate_pack")
DATA = ROOT / "data"
LOGS = ROOT / "logs"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 2026
MAX_LENGTH = 4096
BATCH_SIZE = 1
GRAD_ACCUM = 8
EPOCHS = 1
BASE_LR = 1e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
RULES = (ROOT / "SCHEMA.md").read_text(encoding="utf-8")


def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def require_t4():
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required; select the free Colab T4 runtime.")
    name = torch.cuda.get_device_name(0)
    if "T4" not in name:
        raise RuntimeError(f"Expected a T4 GPU, found {name!r}.")
    if torch.cuda.device_count() != 1:
        raise RuntimeError(f"Expected one T4 GPU, found {torch.cuda.device_count()} devices.")
    print(f"GPU: {name}; memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB")


def model_and_tokenizer(four_bit):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    settings = {"device_map": {"": 0}, "torch_dtype": torch.float16, "attn_implementation": "sdpa"}
    if four_bit:
        settings["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **settings)
    model.config.use_cache = False
    return model, tokenizer


def encode_training_row(row, tokenizer):
    messages = row["messages"]
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=True, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
    if full[:len(prompt)] != prompt:
        raise ValueError(f"Chat template prefix mismatch in {row['id']}")
    if len(full) > MAX_LENGTH:
        raise ValueError(f"{row['id']} is {len(full)} tokens; choose a documented sequence strategy")
    labels = [-100] * len(prompt) + full[len(prompt):]
    if not any(token != -100 for token in labels):
        raise ValueError(f"No assistant tokens supervised in {row['id']}")
    return {"id": row["id"], "input_ids": full, "labels": labels}


def length_report(rows, tokenizer):
    lengths = sorted(len(tokenizer.apply_chat_template(row["messages"], tokenize=True)) for row in rows)
    out = {"n": len(lengths), "median": lengths[len(lengths)//2],
           "p95": lengths[math.ceil(.95*len(lengths))-1],
           "p99": lengths[math.ceil(.99*len(lengths))-1], "max": lengths[-1],
           "over_limit": sum(length > MAX_LENGTH for length in lengths)}
    print("Training sequence lengths:", json.dumps(out))
    return out


def make_batch(item, tokenizer):
    return {
        "input_ids": torch.tensor([item["input_ids"]], dtype=torch.long, device="cuda"),
        "attention_mask": torch.ones((1, len(item["input_ids"])), dtype=torch.long, device="cuda"),
        "labels": torch.tensor([item["labels"]], dtype=torch.long, device="cuda"),
    }


def train(run, output_dir):
    configs = {
        "A": {"dataset": "train_clean.jsonl", "four_bit": True, "lr": BASE_LR, "fraud_extra": False},
        "B": {"dataset": "train_raw_safe.jsonl", "four_bit": True, "lr": BASE_LR, "fraud_extra": False},
        "C": {"dataset": "train_clean.jsonl", "four_bit": False, "lr": BASE_LR, "fraud_extra": False},
        "D": {"dataset": "train_clean.jsonl", "four_bit": True, "lr": 5e-5, "fraud_extra": False},
        "E": {"dataset": "train_clean.jsonl", "four_bit": True, "lr": BASE_LR, "fraud_extra": True},
    }
    config = configs[run]
    rows = read_jsonl(DATA / config["dataset"])
    if config["fraud_extra"]:
        rows += [row for row in rows if json.loads(row["messages"][-1]["content"])["category"] == "fraud"]
    rng = random.Random(SEED)
    rng.shuffle(rows)

    model, tokenizer = model_and_tokenizer(config["four_bit"])
    if config["four_bit"]:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
    ))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")
    if trainable > 50_000_000:
        raise RuntimeError("Adapter exceeds the 50M trainable-parameter limit")

    lengths = length_report(rows, tokenizer)
    encoded = [encode_training_row(row, tokenizer) for row in rows]
    print("Mask check:", encoded[0]["id"], "prompt tokens", encoded[0]["labels"].count(-100),
          "assistant tokens", sum(x != -100 for x in encoded[0]["labels"]))
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=config["lr"])
    updates = math.ceil(len(encoded) / GRAD_ACCUM) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(.03 * updates)), updates)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    losses = []
    update = 0
    for epoch in range(EPOCHS):
        for index, item in enumerate(encoded):
            batch = make_batch(item, tokenizer)
            loss = model(**batch).loss
            (loss / GRAD_ACCUM).backward()
            losses.append(float(loss.detach()))
            if (index + 1) % GRAD_ACCUM == 0 or index + 1 == len(encoded):
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                update += 1
                if update % 25 == 0:
                    print(f"update {update}/{updates}; mean recent loss {sum(losses[-GRAD_ACCUM*25:])/len(losses[-GRAD_ACCUM*25:]):.4f}", flush=True)
    train_minutes = (time.perf_counter() - started) / 60
    peak_gib = torch.cuda.max_memory_allocated() / 2**30
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    metrics = {"run": run, "config": config, "model_id": MODEL_ID, "seed": SEED,
               "max_length": MAX_LENGTH, "epochs": EPOCHS, "grad_accum": GRAD_ACCUM,
               "trainable_parameters": trainable, "lengths": lengths,
               "training_minutes": train_minutes, "peak_gpu_memory_gib": peak_gib}
    return model, tokenizer, metrics


def predict(model, tokenizer, rows, rules_prompt=False):
    model.eval()
    model.config.use_cache = True
    outputs = []
    for index, row in enumerate(rows, 1):
        messages = [dict(message) for message in row["messages"][:2]]
        if rules_prompt:
            messages[0]["content"] += "\n\nFollow these labelling rules exactly:\n" + RULES
        input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                                   return_tensors="pt").to("cuda")
        with torch.inference_mode():
            generated = model.generate(input_ids=input_ids, do_sample=False, max_new_tokens=200,
                                       pad_token_id=tokenizer.eos_token_id)
        raw = tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True).strip()
        outputs.append({"id": row["id"], "output": raw})
        if index % 50 == 0:
            print(f"Generated {index}/{len(rows)}", flush=True)
    return outputs



## Baselines and controlled experiments


In [ ]:
def run_dev(run):
    require_t4()
    seed_all(SEED)
    run_dir = LOGS / run
    run_dir.mkdir(parents=True, exist_ok=True)
    if run in {'A', 'B', 'C', 'D', 'E'}:
        model, tokenizer, metrics = train(run, run_dir / 'adapter')
        (run_dir / 'adapter' / 'run_config.json').write_text(
            json.dumps({'run': run, 'four_bit': run != 'C'}, indent=2))
    else:
        model, tokenizer = model_and_tokenizer(True)
        metrics = {'run': run, 'model_id': MODEL_ID, 'four_bit': True}
    dev = read_jsonl(DATA / 'dev.jsonl')
    outputs = predict(model, tokenizer, dev, rules_prompt=run == 'base_rules')
    write_jsonl(run_dir / 'dev_predictions.jsonl', outputs)
    metrics['dev_score'] = score(dev, outputs)
    (run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    print(run, metrics['dev_score']['all'])
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

RUN_EXPERIMENTS = False
SELECTED_RUN = 'A'
assert SELECTED_RUN in {'A', 'B', 'C', 'D', 'E'}
LOGS.mkdir(exist_ok=True)
if RUN_EXPERIMENTS:
    for run in ['base_fixed', 'base_rules', 'A', 'B', 'C', 'D', 'E']:
        backup_run = Path('/content/drive/MyDrive/nimbuspay_output/logs') / run
        if (backup_run / 'metrics.json').exists():
            shutil.copytree(backup_run, LOGS / run, dirs_exist_ok=True)
            print('Restored:', run)
        else:
            run_dev(run)
            shutil.copytree(LOGS / run, backup_run, dirs_exist_ok=True)
    candidates = {}
    for run in ['A', 'B', 'C', 'D', 'E']:
        overall = json.loads((LOGS / run / 'metrics.json').read_text())['dev_score']['all']
        candidates[run] = (overall['exact_match'], overall['mean_field_acc'])
    SELECTED_RUN = max(candidates, key=candidates.get)
    print('Selected:', SELECTED_RUN, candidates[SELECTED_RUN])
else:
    run_dev(SELECTED_RUN)


In [ ]:
for run in ['base_fixed', 'base_rules', 'A', 'B', 'C', 'D', 'E']:
    path = LOGS / run / 'metrics.json'
    if path.exists():
        metric = json.loads(path.read_text())
        overall = metric['dev_score']['all']
        print(run, overall['exact_match'], overall['mean_field_acc'],
              metric.get('peak_gpu_memory_gib'), metric.get('training_minutes'))


## Final adapter and test predictions


In [ ]:
source_adapter = LOGS / SELECTED_RUN / 'adapter'
final_adapter = WORK / 'adapter'
shutil.copytree(source_adapter, final_adapter, dirs_exist_ok=True)
model, tokenizer = model_and_tokenizer(SELECTED_RUN != 'C')
model = PeftModel.from_pretrained(model, final_adapter)
test_outputs = predict(model, tokenizer, read_jsonl(DATA / 'test_inputs.jsonl'))
write_jsonl(WORK / 'predictions_test.jsonl', test_outputs)
assert len(test_outputs) == 400 and len({row['id'] for row in test_outputs}) == 400
assert (final_adapter / 'adapter_config.json').exists()
assert (final_adapter / 'adapter_model.safetensors').exists()
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

BACKUP = Path('/content/drive/MyDrive/nimbuspay_output')
BACKUP.mkdir(exist_ok=True)
for name in ['adapter', 'logs']:
    shutil.copytree(WORK / name, BACKUP / name, dirs_exist_ok=True)
for name in ['cleaning_log.csv', 'predictions_test.jsonl']:
    shutil.copy2(WORK / name, BACKUP / name)
print('Saved', len(test_outputs), 'test predictions and adapter to', BACKUP)
